# Data Preparation for GRPO Text-to-SQL Training

This notebook prepares the training corpus from two sources:

1. **BIRD-Platinum** (2,064 expert-verified text-to-SQL pairs from [ReViSQL](https://github.com/ThinkingMachinesLab/ReViSQL))
2. **NNDSS domain pairs** (~300 generated question-SQL pairs against the Trino lakehouse)

Outputs are formatted for SFT warmup and GRPO training, then uploaded to the shared PVC.

## Setup

In [ ]:
%env UV_EXTRA_INDEX_URL=https://pypi.org/simple
!uv pip install pandas pyarrow fastparquet trino gdown huggingface_hub

In [ ]:
import json
import os
import random
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data")
OUTPUT_DIR = DATA_DIR
OUTPUT_DIR.mkdir(exist_ok=True)

SEED = 42
random.seed(SEED)

## 1. Load BIRD-Platinum

BIRD-Platinum is the expert-verified training set from the ReViSQL paper.
No LLM cleaning pass is needed — the data has already been curated.

In [ ]:
bird_train = pd.read_parquet(DATA_DIR / "bird-verified-train.parquet")
bird_val = pd.read_parquet(DATA_DIR / "val_test.parquet")

print(f"BIRD-Platinum training: {len(bird_train)} rows")
print(f"BIRD-Platinum val/eval: {len(bird_val)} rows")
print(f"\nColumns: {list(bird_train.columns)}")

In [ ]:
# Inspect one example
row = bird_train.iloc[0]
print("=== Sample BIRD-Platinum example ===")
print(f"\ndb_id: {row['db_id']}")
print(f"question_id: {row['question_id']}")
print(f"\nPrompt messages ({len(row['prompt'])} turns):")
for msg in row["prompt"]:
    print(f"  [{msg['role']}] {msg['content'][:200]}...")
print(f"\nReward spec:")
print(f"  gold SQL: {row['reward_spec']['ground_truth'][:200]}")
print(f"  grading method: {row['reward_spec']['method']}")

## 2. Format BIRD-Platinum for Training

Convert the parquet prompt format to Training Hub's expected chat message format.

In [ ]:
def bird_row_to_sft(row):
    """Convert a BIRD-Platinum row to SFT chat format."""
    messages = list(row["prompt"])  # system + user messages
    gold_sql = row["reward_spec"]["ground_truth"]
    messages.append({"role": "assistant", "content": gold_sql})
    return {"messages": messages}


def bird_row_to_grpo(row):
    """Convert a BIRD-Platinum row to GRPO prompt format.

    Training Hub's verl backend expects 'messages' and 'ground_truth' fields.
    'data_source' and 'extra_info' are passed through to the reward function
    so it can execute SQL against the correct SQLite database.
    """
    return {
        "messages": list(row["prompt"]),
        "ground_truth": row["reward_spec"]["ground_truth"],
        "data_source": "bird",
        "extra_info": {
            "db_id": row["db_id"],
            "db_type": "bird",
            "grading_method": row["reward_spec"].get("method", "set"),
        },
    }


bird_sft_data = [bird_row_to_sft(row) for _, row in bird_train.iterrows()]
bird_grpo_data = [bird_row_to_grpo(row) for _, row in bird_train.iterrows()]

print(f"BIRD SFT examples: {len(bird_sft_data)}")
print(f"BIRD GRPO prompts: {len(bird_grpo_data)}")
print(f"  Sample extra_info: {bird_grpo_data[0]['extra_info']}")

## 3. Generate NNDSS Domain Pairs

Generate question-SQL pairs from the NNDSS schema using templates.
These pairs teach the model Trino-specific SQL patterns for the public health domain.

In [ ]:
import sys
sys.path.insert(0, str(Path("../").resolve()))

from reward.nndss_schema import (
    NNDSS_DDL, DISEASES, STATES, YEARS, SQL_PATTERNS,
)

nndss_pairs = []

for pattern_name, (q_template, sql_template) in SQL_PATTERNS.items():
    for disease in DISEASES:
        for state in random.sample(STATES, min(3, len(STATES))):
            for year in random.sample(YEARS[-5:], 2):  # recent years
                try:
                    if "year1" in q_template:
                        year1, year2 = year - 3, year
                        question = q_template.format(
                            disease=disease, state=state,
                            year=year, year1=year1, year2=year2,
                        )
                        sql = sql_template.format(
                            disease=disease, state=state,
                            year=year, year1=year1, year2=year2,
                        )
                    else:
                        question = q_template.format(
                            disease=disease, state=state, year=year,
                        )
                        sql = sql_template.format(
                            disease=disease, state=state, year=year,
                        )
                    nndss_pairs.append({
                        "question": question,
                        "sql": sql,
                        "pattern": pattern_name,
                    })
                except (KeyError, IndexError):
                    continue

# Deduplicate and sample
seen = set()
unique_pairs = []
for p in nndss_pairs:
    key = p["sql"]
    if key not in seen:
        seen.add(key)
        unique_pairs.append(p)

random.shuffle(unique_pairs)
nndss_pairs = unique_pairs[:300]

print(f"Generated {len(nndss_pairs)} unique NNDSS pairs")
print(f"\nPattern distribution:")
from collections import Counter
for pattern, count in Counter(p["pattern"] for p in nndss_pairs).most_common():
    print(f"  {pattern}: {count}")

### Validate NNDSS Pairs Against Trino

Execute each generated SQL against the live Trino database to verify correctness.

**First-time setup:** Deploy Trino + NNDSS data with:
```bash
cd /path/to/rl-sql && ./deploy/deploy-trino.sh
```

Then port-forward: `oc port-forward svc/trino 8090:8080 -n rl-sql`

In [ ]:
VALIDATE_AGAINST_TRINO = True  # Set to False to skip validation
TRINO_HOST = "trino"  # in-cluster service name
TRINO_PORT = 8080     # in-cluster port (use 8090 only with local port-forward)

if VALIDATE_AGAINST_TRINO:
    try:
        from trino.dbapi import connect as trino_connect
    except ImportError:
        !pip install trino
        from trino.dbapi import connect as trino_connect

    print(f"Connecting to Trino at {TRINO_HOST}:{TRINO_PORT}...")
    conn = trino_connect(
        host=TRINO_HOST, port=TRINO_PORT, user="admin",
        catalog="lakehouse", schema="nndss",
    )

    # Quick connectivity check
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM lakehouse.nndss.notifications")
    print(f"Connected OK — notifications table has {cur.fetchone()[0]} rows\n")

    valid_pairs = []
    errors = []
    total = len(nndss_pairs)
    for i, pair in enumerate(nndss_pairs):
        try:
            cur = conn.cursor()
            cur.execute(pair["sql"])
            results = cur.fetchall()
            if results:
                valid_pairs.append(pair)
            else:
                errors.append((pair["sql"][:80], "empty results"))
        except Exception as e:
            errors.append((pair["sql"][:80], str(e)[:80]))

        if (i + 1) % 10 == 0 or (i + 1) == total:
            print(
                f"  [{i+1}/{total}] "
                f"valid: {len(valid_pairs)}, errors: {len(errors)}, "
                f"pattern: {pair.get('pattern', '?')}"
            )

    conn.close()
    nndss_pairs = valid_pairs
    print(f"\nValid NNDSS pairs: {len(valid_pairs)}")
    print(f"Errors: {len(errors)}")
    if errors:
        print("\nFirst 5 errors:")
        for sql, err in errors[:5]:
            print(f"  {sql}... -> {err}")
else:
    print(f"Skipping Trino validation. {len(nndss_pairs)} pairs kept as-is.")

### Format NNDSS Pairs for Training

In [ ]:
NNDSS_SYSTEM_PROMPT = (
    "You are a SQL expert. Given a database schema, write a SQL query "
    "that answers the user's question. Output only the SQL query."
)


def nndss_pair_to_sft(pair):
    return {
        "messages": [
            {"role": "system", "content": NNDSS_SYSTEM_PROMPT},
            {"role": "user", "content": f"Schema:\n{NNDSS_DDL}\n\nQuestion: {pair['question']}"},
            {"role": "assistant", "content": pair["sql"]},
        ]
    }


def nndss_pair_to_grpo(pair):
    """Training Hub verl backend expects 'messages' and 'ground_truth'.
    'data_source' and 'extra_info' route the reward to Trino execution."""
    return {
        "messages": [
            {"role": "system", "content": NNDSS_SYSTEM_PROMPT},
            {"role": "user", "content": f"Schema:\n{NNDSS_DDL}\n\nQuestion: {pair['question']}"},
        ],
        "ground_truth": pair["sql"],
        "data_source": "nndss",
        "extra_info": {
            "db_type": "nndss",
            "grading_method": "set",
        },
    }


nndss_sft_data = [nndss_pair_to_sft(p) for p in nndss_pairs]
nndss_grpo_data = [nndss_pair_to_grpo(p) for p in nndss_pairs]

print(f"NNDSS SFT examples: {len(nndss_sft_data)}")
print(f"NNDSS GRPO prompts: {len(nndss_grpo_data)}")

## 4. Combine and Save

Merge BIRD-Platinum and NNDSS data, split into train/val, and save.

In [ ]:
# Combine SFT data
all_sft = bird_sft_data + nndss_sft_data
random.shuffle(all_sft)

# Combine GRPO data — 90/10 train/val split
all_grpo = bird_grpo_data + nndss_grpo_data
random.shuffle(all_grpo)
split_idx = int(len(all_grpo) * 0.9)
grpo_train = all_grpo[:split_idx]
grpo_val = all_grpo[split_idx:]

# Save SFT data
sft_path = OUTPUT_DIR / "sft_train_data.jsonl"
with open(sft_path, "w") as f:
    for ex in all_sft:
        f.write(json.dumps(ex) + "\n")

# Save GRPO data
grpo_train_path = OUTPUT_DIR / "grpo_prompts.jsonl"
with open(grpo_train_path, "w") as f:
    for ex in grpo_train:
        f.write(json.dumps(ex) + "\n")

grpo_val_path = OUTPUT_DIR / "grpo_val_prompts.jsonl"
with open(grpo_val_path, "w") as f:
    for ex in grpo_val:
        f.write(json.dumps(ex) + "\n")

# Save NNDSS pairs for reference
nndss_path = OUTPUT_DIR / "nndss_pairs" / "nndss_pairs.jsonl"
nndss_path.parent.mkdir(exist_ok=True)
with open(nndss_path, "w") as f:
    for p in nndss_pairs:
        f.write(json.dumps(p) + "\n")

print("=== Data Summary ===")
print(f"SFT training examples: {len(all_sft)} -> {sft_path}")
print(f"  BIRD-Platinum: {len(bird_sft_data)}")
print(f"  NNDSS:         {len(nndss_sft_data)}")
print(f"GRPO train prompts:    {len(grpo_train)} -> {grpo_train_path}")
print(f"GRPO val prompts:      {len(grpo_val)} -> {grpo_val_path}")
print(f"NNDSS pairs saved:     {len(nndss_pairs)} -> {nndss_path}")

## 5. Download BIRD SQLite Databases

The BIRD SQLite databases are needed on the PVC for execution-based rewards during GRPO training.
Run the download script, then use `upload_to_pvc.sh` to copy everything to the shared PVC.

In [ ]:
import shutil, subprocess, zipfile

bird_db_dir = DATA_DIR / "bird_databases"
db_count = len(list(bird_db_dir.rglob("*.sqlite"))) if bird_db_dir.exists() else 0

if db_count >= 11:
    print(f"BIRD databases present: {db_count} databases in {bird_db_dir}")
else:
    print(f"Only {db_count} BIRD databases found — downloading from bird-bench...")
    dl_dir = Path("/tmp/bird_dl")
    zip_path = Path("/tmp/minidev.zip")
    dl_dir.mkdir(exist_ok=True)
    subprocess.run(
        ["curl", "-L", "-o", str(zip_path),
         "https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip"],
        check=True,
    )
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(dl_dir)
    src = dl_dir / "minidev" / "MINIDEV" / "dev_databases"
    bird_db_dir.mkdir(parents=True, exist_ok=True)
    for db_dir in src.iterdir():
        dest = bird_db_dir / db_dir.name
        if not dest.exists():
            shutil.copytree(db_dir, dest)
    shutil.rmtree(dl_dir, ignore_errors=True)
    zip_path.unlink(missing_ok=True)
    db_count = len(list(bird_db_dir.rglob("*.sqlite")))
    print(f"Downloaded {db_count} BIRD databases to {bird_db_dir}")

## 6. Upload to Shared PVC

Copy training data and reward module to the shared PVC so Ray workers can access them during training.

In [ ]:
import shutil

PVC_MOUNT = "/opt/app-root/src/shared"
TARGET = os.path.join(PVC_MOUNT, "text2sql")

os.makedirs(os.path.join(TARGET, "reward"), exist_ok=True)

# Make dirs writable by Ray pods — skip if already owned by another UID
for subdir in ["", "sft_checkpoint", "grpo_output"]:
    d = os.path.join(TARGET, subdir) if subdir else TARGET
    os.makedirs(d, exist_ok=True)
    try:
        os.chmod(d, 0o777)
    except PermissionError:
        print(f"  Skipping chmod on {d} (owned by Ray pod, already writable)")

# Copy training data
for f in ["sft_train_data.jsonl", "grpo_prompts.jsonl", "grpo_val_prompts.jsonl"]:
    src = OUTPUT_DIR / f
    if src.exists():
        shutil.copy2(src, os.path.join(TARGET, f))
        print(f"  Copied {f} ({src.stat().st_size // 1024}KB)")

# Copy NNDSS pairs
nndss_src = OUTPUT_DIR / "nndss_pairs" / "nndss_pairs.jsonl"
if nndss_src.exists():
    os.makedirs(os.path.join(TARGET, "nndss_pairs"), exist_ok=True)
    shutil.copy2(nndss_src, os.path.join(TARGET, "nndss_pairs", "nndss_pairs.jsonl"))
    print(f"  Copied nndss_pairs.jsonl")

# Copy reward module and patch imports for standalone loading.
# verl loads verl_reward.py directly (not as a package), so sibling
# files must use flat imports ("from grader import grade") not
# package imports ("from reward.grader import grade").
reward_dir = Path("../reward")
for f in reward_dir.glob("*.py"):
    dest = os.path.join(TARGET, "reward", f.name)
    shutil.copy2(f, dest)
    with open(dest) as fh:
        content = fh.read()
    patched = content.replace("from reward.grader import", "from grader import")
    patched = patched.replace("from reward.sql_executor import", "from sql_executor import")
    patched = patched.replace("from reward.nndss_schema import", "from nndss_schema import")
    patched = patched.replace("from reward.reward_fn import", "from reward_fn import")
    if patched != content:
        with open(dest, "w") as fh:
            fh.write(patched)
        print(f"  Copied + patched reward/{f.name}")
    else:
        print(f"  Copied reward/{f.name}")

# Copy BIRD databases if downloaded
bird_src = OUTPUT_DIR / "bird_databases"
bird_dst = os.path.join(TARGET, "bird_databases")
if bird_src.exists() and any(bird_src.rglob("*.sqlite")):
    if not os.path.exists(bird_dst):
        print(f"  Copying BIRD databases (this may take a moment)...")
        shutil.copytree(bird_src, bird_dst)
    else:
        print(f"  BIRD databases already on PVC")
    db_count = len(list(Path(bird_dst).rglob("*.sqlite")))
    print(f"  BIRD databases: {db_count} .sqlite files")
else:
    print(f"  Skipping BIRD databases (not downloaded yet)")

print(f"\n=== PVC contents at {TARGET} ===")
for item in sorted(Path(TARGET).rglob("*")):
    if item.is_file() and ".sqlite" not in str(item):
        rel = item.relative_to(TARGET)
        print(f"  {rel} ({item.stat().st_size // 1024}KB)")